# Parte 1 — O primeiro modelo em produção

**Banco Aurora · Prevenção a Fraudes** · entrega: **quarta-feira, 11/11, 23h59**

Leia antes o `demanda_parte1.md`: é o pedido do negócio. Este notebook é o roteiro técnico para atendê-lo.

**Como este notebook funciona**
- Cada seção diz **o que** fazer e **por quê**. O código é seu.
- Duas células vêm prontas (seção 0 e seção 10): elas são encanamento, não conteúdo.
- Tudo que será reutilizado nas partes 2 e 3 vai para arquivos `.py` (`features.py`, `monitoramento.py`), não fica só no notebook.
- Rode a partir da raiz do projeto, com o **MLflow** e o **sistema** no ar (veja o `README.md`).

**O que será avaliado nesta parte**
1. O contrato de features é respeitado no treino e na inferência.
2. O modelo está registrado e rastreável no MLflow, como `@champion`.
3. O threshold é justificado pelo custo de negócio.
4. O monitoramento está pronto **antes** do primeiro lote chegar.

O algoritmo é escolha sua. **Não existe ranking de performance**: existe um piso (seção 5) e existe a qualidade da justificativa.

## 0. Ambiente

Rode a célula abaixo. Se algo aparecer como `FALHOU`, resolva antes de seguir: o resto do notebook depende das quatro conexões.

In [ ]:
# CÉLULA PRONTA — só rode. Tudo precisa aparecer como "ok".
import json
import urllib.request

import mlflow
import psycopg2

import config

for nome, uri in [("Postgres / antifraude", config.URI_BANCO_APP), ("Postgres / mlflow", config.URI_BANCO_MLFLOW)]:
    try:
        psycopg2.connect(uri).close()
        print(f"{nome:24} ok")
    except Exception as e:
        print(f"{nome:24} FALHOU -> {e}")

for nome, url in [("MLflow", f"{config.MLFLOW_TRACKING_URI}/health"), ("Sistema", f"{config.SISTEMA_URL}/saude")]:
    try:
        resp = urllib.request.urlopen(url, timeout=5).read().decode()
        print(f"{nome:24} ok -> {resp[:160]}")
    except Exception as e:
        print(f"{nome:24} FALHOU -> {e}")

mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
print("\ntracking uri:", mlflow.get_tracking_uri())

## 1. O contrato

O contrato (`contrato/contrato_features.yaml`) é o acordo com o time de Plataforma: as 13 features que o sistema de produção entrega, com tipo, regra de cálculo e valores permitidos. **O sistema rejeita qualquer lote que não o respeite**, e o seu modelo vai receber exatamente isso — nada mais.

**Faça**
1. Leia o YAML com `yaml.safe_load` e monte, a partir dele (não digitando à mão), a lista das features na ordem do contrato, separando categóricas e numéricas.
2. Carregue `dados/transacoes_semanas_01_12.csv` e `dados/rotulos_semanas_01_12.csv` e junte os dois pela `id_transacao`. Garanta que a junção é 1 para 1 e que nenhuma linha se perdeu.
3. Escreva uma validação do histórico contra o contrato: colunas presentes, tipos, valores fora da lista nas categóricas, valores fora de `min`/`max`, nulos. O histórico passa?

> Por que validar um dado que "veio do próprio banco"? Porque o modelo vai aprender o que estiver aqui. Se o treino tiver algo que a produção não terá (ou o contrário), o erro só aparece depois, em produção.

## 2. Divisão temporal

A regra combinada com o negócio: **treino nas semanas 1 a 10, validação nas semanas 11 e 12**.

**Faça**
1. Separe os dois conjuntos pela coluna `semana`.
2. Mostre a taxa de fraude por semana e o tamanho de cada conjunto.
3. Responda em uma célula de texto: por que uma divisão aleatória (`train_test_split`) seria um erro aqui?

*Sua resposta:*

## 3. Pré-processamento em `features.py`

Escreva no arquivo `features.py` (não no notebook) o que transforma as 13 features cruas na entrada do seu algoritmo. Depois, importe aqui.

**Requisitos**
- O pré-processamento precisa ser parte do modelo que você registra (um `Pipeline` do scikit-learn). Em produção, o sistema entrega as features cruas e chama `predict_proba` — ele não transforma nada.
- As listas de colunas e de categorias vêm **do contrato**, não do que apareceu no treino. Pense no que acontece se uma UF que não está no treino aparecer em produção.
- Se o seu pipeline usar classes ou funções suas, o modelo precisa ser registrado com `code_paths=["features.py"]`.

> Dica: depois de editar o `features.py`, use `importlib.reload(features)` ou reinicie o kernel.

## 4. Pelo menos dois modelos, tudo no MLflow

Treine **pelo menos dois algoritmos** diferentes, cada um em uma run do MLflow.

**Faça**
1. Crie (ou use) um experimento com nome que você reconheça depois.
2. Em cada run, registre:
   - **parâmetros**: algoritmo, hiperparâmetros relevantes, semanas de treino e de validação, versão do contrato;
   - **métricas na validação**: PR-AUC, recall, precision e **custo** (seção 5);
   - **o modelo** (`mlflow.sklearn.log_model`), com assinatura (`infer_signature`) e um `input_example`.
3. Compare as runs na interface do MLflow.

> Use PR-AUC e não ROC-AUC para comparar ranking: com 1,5% de fraude, a ROC-AUC fica alta até para modelos ruins.

## 5. Threshold pelo custo

O `predict_proba` devolve um score. O **threshold** é o corte: `score >= threshold` → bloquear. O padrão de 0,5 não tem nada a ver com o problema do banco.

**Custo de um conjunto de decisões** = soma do **valor** das fraudes aprovadas + **R$ 15** × legítimas bloqueadas.

**Faça**
1. Escreva a função de custo (ela vai para o `monitoramento.py` na seção 9; pode começar aqui).
2. Para cada modelo, varra thresholds de 0,01 a 0,99 na **validação** e escolha o de menor custo. Faça o gráfico custo × threshold.
3. No threshold escolhido, mostre recall, precision e custo. Registre o threshold e o custo como métricas da run.
4. Mostre a comparação com o **piso** pedido pelo negócio:
   - aprovar tudo;
   - a regra atual: bloquear todo Pix acima de R$ 1.000;
   - o próprio modelo com threshold 0,5.

## 6. A escolha

Em uma célula de texto, escolha o modelo que vai para produção e justifique **em linguagem de custo**, como você explicaria para a Helena: quanto ele custa por semana na validação, quanto economiza em relação à regra atual, quantos clientes legítimos ele incomoda e quanta fraude ainda passa.

Não vale "porque teve a maior PR-AUC". Vale usar a PR-AUC como argumento de apoio.

*Sua justificativa:*

## 7. Registro e promoção

**Faça**
1. Registre o modelo escolhido no Model Registry com o nome **`antifraude`** (é o nome que o sistema procura).
2. Na **versão** registrada (não na run), grave as tags:
   - `threshold` — o corte escolhido na seção 5 (o sistema lê daqui);
   - `motivo` — uma frase curta, ex.: `v1.0.0 - primeiro modelo em produção` (aparece no histórico de trocas do sistema).
3. Escreva uma **descrição** para a versão: algoritmo, threshold e por que foi escolhido.
4. Aponte o alias **`champion`** para essa versão.

Use o `MlflowClient` para os passos 2 a 4.

## 8. O sistema está usando o seu modelo?

Abra http://127.0.0.1:8000/modelo: deve aparecer a sua versão e o seu threshold.

**Faça**
1. Pegue 5 transações da semana 12 (só as 13 features) e mande para `POST /predict` do sistema (lista de objetos JSON).
2. Pontue as mesmas 5 transações com o modelo carregado do MLflow (`models:/antifraude@champion`) no notebook.
3. Confira que os scores são **idênticos** e que a decisão (bloquear ou não) bate com o seu threshold.

> Se os scores forem diferentes, há alguma transformação acontecendo de um jeito no notebook e de outro em produção. É exatamente o tipo de bug que o contrato existe para evitar.

## 9. Monitoramento baseline

A partir de quarta-feira chegam as semanas novas, uma por vez, e os rótulos de cada semana chegam só uma semana depois. O seu monitoramento precisa estar pronto **antes**, para que você perceba problemas sem esperar o negócio reclamar.

### 9.1 Referência
A referência é a "foto" de como os dados e os scores eram quando o modelo foi aprovado: as features da **validação** (semanas 11 e 12), o score do modelo escolhido e o rótulo.
- Salve a referência em arquivo e registre-a como **artefato da run do modelo campeão**, na pasta `referencia/` (`mlflow.log_artifacts(..., artifact_path="referencia", run_id=...)`). Assim, trocou o modelo, trocou a referência.

### 9.2 `monitoramento.py`
Escreva funções reutilizáveis que recebam a referência e um lote e devolvam tabelas legíveis:
- **Drift das features contínuas e do score**: teste **KS**, reportando a **estatística D** e o p-valor.
- **Drift das categóricas e booleanas** (e da hora): qui-quadrado ou comparação de proporções.
- **Saída do modelo**: taxa de bloqueio e valor bloqueado.
- **Performance, para quando o rótulo chegar**: recall, precision, PR-AUC e custo — **globais e por segmento** (uma coluna à sua escolha, ex.: `canal`).

### 9.3 Relatório zero
Rode o monitoramento comparando a **semana 11 com a semana 12** (as duas são da validação, nada mudou de propósito entre elas).
- Olhe os p-valores e as estatísticas. Algum p-valor é minúsculo mesmo sem nenhuma mudança relevante? Por quê?
- Defina e **justifique por escrito** os seus limites de alerta (ex.: D > 0,1). Eles valem para as próximas partes.

> O que você vai receber a cada semana: o CSV de transações (que você envia em **Lotes** no sistema) e, a partir da segunda semana, o CSV de rótulos da semana anterior (em **Rótulos**). Os dados já pontuados ficam em `GET /api/semanas/{n}/dados.csv`, com o score e, quando houver, o rótulo. É daí que o seu monitoramento vai ler.

*Seus limites de alerta e a justificativa:*

## 10. Entrega

1. Escreva a entrada **v1.0.0** no `CHANGELOG.md`, seguindo o modelo do próprio arquivo.
2. Confira a checklist:
   - [ ] `antifraude@champion` aponta para o modelo escolhido, com tags `threshold` e `motivo` e descrição;
   - [ ] a run do champion tem a referência em `referencia/`;
   - [ ] o sistema mostra a sua versão em **Modelo**, e o teste da seção 8 bateu;
   - [ ] `features.py` e `monitoramento.py` têm o seu código;
   - [ ] as células de texto (seções 2, 6 e 9) estão respondidas.
3. Preencha o seu nome, **salve o notebook** e rode a célula abaixo. Ela gera a pasta `entrega/` (evidências do MLflow, modelo exportado, referência, seus arquivos) e o `entrega_parte1_<nome>.zip`.
4. Entregue o `.zip` na atividade. Se a célula listar **avisos**, resolva antes de entregar.

> A célula não mexe em nada no MLflow nem no sistema: ela só lê e copia.

In [ ]:
# CÉLULA PRONTA — gera a pasta entrega/ e o entrega_parte1_<nome>.zip.
# 1) Preencha o seu nome.  2) SALVE o notebook (Ctrl+S).  3) Rode.
NOME_ALUNO = ""

import datetime
import json
import shutil
import urllib.request
from pathlib import Path

import mlflow
from mlflow.tracking import MlflowClient

import config

assert NOME_ALUNO.strip(), "Preencha NOME_ALUNO antes de rodar."
mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
cli = MlflowClient()
mv = cli.get_model_version_by_alias(config.NOME_MODELO, config.ALIAS_PRODUCAO)
run = cli.get_run(mv.run_id)
art_raiz = [a.path for a in cli.list_artifacts(mv.run_id)]
art_ref = [a.path for a in cli.list_artifacts(mv.run_id, "referencia")]
saude = json.loads(urllib.request.urlopen(f"{config.SISTEMA_URL}/saude", timeout=5).read())

avisos = []
if config.TAG_THRESHOLD not in mv.tags:
    avisos.append("a versão champion não tem a tag 'threshold'")
if not mv.description:
    avisos.append("a versão champion não tem descrição")
if config.TAG_MOTIVO not in mv.tags:
    avisos.append("a versão champion não tem a tag 'motivo'")
if not art_ref:
    avisos.append("a run do champion não tem artefatos em 'referencia/'")
if str(saude["modelo"].get("versao")) != str(mv.version):
    avisos.append("o sistema não está usando a mesma versão que o MLflow aponta como champion")
for arq in ["features.py", "monitoramento.py"]:
    linhas = [l for l in Path(arq).read_text(encoding="utf-8").splitlines() if l.strip() and not l.lstrip().startswith("#")]
    if sum(1 for l in linhas if l.lstrip().startswith(("def ", "class ", "import ", "from "))) == 0:
        avisos.append(f"{arq} parece vazio")
if "## v1.0.0" not in Path("CHANGELOG.md").read_text(encoding="utf-8"):
    avisos.append("CHANGELOG.md não tem a entrada v1.0.0")

evidencias = {
    "aluno": NOME_ALUNO.strip(),
    "gerado_em": datetime.datetime.now().isoformat(timespec="seconds"),
    "modelo": config.NOME_MODELO, "alias": config.ALIAS_PRODUCAO,
    "versao": mv.version, "descricao": mv.description, "tags_versao": dict(mv.tags),
    "run_id": mv.run_id, "experimento": run.info.experiment_id,
    "params": run.data.params, "metricas": run.data.metrics,
    "artefatos": art_raiz, "referencia": art_ref,
    "sistema": saude, "avisos": avisos,
}

pasta = Path("entrega")
shutil.rmtree(pasta, ignore_errors=True)
pasta.mkdir()
(pasta / "evidencias_mlflow.json").write_text(json.dumps(evidencias, indent=2, ensure_ascii=False), encoding="utf-8")
mlflow.artifacts.download_artifacts(artifact_uri=f"models:/{config.NOME_MODELO}@{config.ALIAS_PRODUCAO}",
                                    dst_path=str(pasta / "modelo_exportado"))
mlflow.artifacts.download_artifacts(run_id=mv.run_id, artifact_path="referencia", dst_path=str(pasta)) if art_ref else None
for arq in ["parte1.ipynb", "features.py", "monitoramento.py", "CHANGELOG.md", "contrato/contrato_features.yaml"]:
    shutil.copy(arq, pasta / Path(arq).name)

nome_zip = "entrega_parte1_" + "_".join(NOME_ALUNO.lower().split())
shutil.make_archive(nome_zip, "zip", pasta)
print(f"champion: {config.NOME_MODELO} v{mv.version} | threshold {mv.tags.get(config.TAG_THRESHOLD)}")
print("AVISOS:" if avisos else "Nenhum aviso.", *avisos, sep="\n  - ")
print(f"\nEntregue o arquivo {nome_zip}.zip")